In [1]:
import os
import json
import pandas as pd
import typing

from pathlib import Path
from typing_extensions import NotRequired, Required
from mp_api.client import MPRester

typing.NotRequired = NotRequired
typing.Required = Required

c:\venvs\battery_gnn\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
API_KEY = os.environ["MP_API_KEY"]

outdir = Path("datos_baterias_mp")
outdir.mkdir(exist_ok=True)

jsonl_file = outdir / "baterias_materials_project.jsonl"
csv_file = outdir / "baterias_materials_project_resumen.csv"

In [5]:
with MPRester(
    API_KEY,
    use_document_model=False,
) as mpr:

    electrodes = (
        mpr.materials.insertion_electrodes
    )

    docs = electrodes.search(
        all_fields=True,
        chunk_size=1000,
    )

    docs = list(docs)

    print(
        f"Documentos descargados: {len(docs)}"
    )

    with open(
        jsonl_file,
        "w",
        encoding="utf-8",
    ) as f:

        for doc in docs:

            f.write(
                json.dumps(
                    doc,
                    default=str,
                    ensure_ascii=False,
                )
                + "\n"
            )

    campos_resumen = [
        "battery_id",
        "battery_formula",
        "working_ion",
        "average_voltage",
        "capacity_grav",
        "capacity_vol",
        "energy_grav",
        "energy_vol",
        "max_delta_volume",
        "max_voltage_step",
        "stability_charge",
        "stability_discharge",
        "framework_formula",
        "num_steps",
        "last_updated",
    ]

    resumen = []

    for doc in docs:

        fila = {
            campo: doc.get(
                campo,
                None,
            )
            for campo in campos_resumen
        }

        resumen.append(
            fila
        )

    df = pd.DataFrame(
        resumen
    )

    df.to_csv(
        csv_file,
        index=False,
    )


print(
    f"JSONL completo guardado en: {jsonl_file}"
)

print(
    f"CSV resumido guardado en: {csv_file}"
)

mp_api.client.core.client - INFO - Dataset for insertion-electrodes written to C:\Users\NereaBorjaGonzález\mp_datasets\build\collections\insertion-electrodes
mp_api.client.core.client - INFO - Converting to DeltaTable...
mp_api.client.core.client - INFO - Consult the delta-rs and pyarrow documentation for advanced usage: delta-io.github.io/delta-rs, arrow.apache.org/docs/python
C:\Users\NereaBorjaGonzález\AppData\Local\Temp\ipykernel_34928\339889348.py:15: MPDatasetIterationWarning: 
                Iterating through arrow-based MPDatasets is sub-optimal, consider using
                idiomatic arrow patterns. See MP's docs on MPDatasets for relevant examples:
                docs.materialsproject.org/materials-project-data-lakehouse/arrow-datasets
                
  docs = list(docs)


Documentos descargados: 6847
JSONL completo guardado en: datos_baterias_mp\baterias_materials_project.jsonl
CSV resumido guardado en: datos_baterias_mp\baterias_materials_project_resumen.csv
